# 3. Inspect and clean trip-level Citi Bike data (2013–2019)

**Who worked on this:** Rune Lemmens

This notebook inspects the assembled trip CSV, reports missingness and numeric ranges, applies the documented cleaning rules, and writes a separate cleaned CSV. Run 02_assemble.ipynb first.

In [2]:
import gzip
from pathlib import Path

import pandas as pd

# Standalone data paths and constants used by Sections 4–5.
DATA_DIR = Path.cwd() / "data"
OUTPUT_PATH = DATA_DIR / "processed" / "citibike_trips_unified_2013_2019.csv"
CLEANED_OUTPUT_PATH = DATA_DIR / "processed" / "citibike_trips_cleaned_2013_2019.csv"
CHUNK_SIZE = 200_000
UNIFIED_COLUMNS = [
    "started_at", "ended_at", "start_station_id", "start_station_name", "start_lat", "start_lng",
    "end_station_id", "end_station_name", "end_lat", "end_lng", "member_casual", "birth_year", "gender", "trip_duration_s",
]
if not OUTPUT_PATH.exists():
    raise FileNotFoundError(f"Unified trip CSV not found: {OUTPUT_PATH}. Run 02_assemble.ipynb first.")

In [3]:
row_count = sum(
    len(chunk)
    for chunk in pd.read_csv(OUTPUT_PATH, usecols=["started_at"], chunksize=CHUNK_SIZE)
)
print(f"Total individual trips in combined CSV: {row_count:,}")
print(f"Output size: {OUTPUT_PATH.stat().st_size / 1_000_000_000:.2f} GB")

sample = pd.read_csv(OUTPUT_PATH, nrows=5)
sample

Total individual trips in combined CSV: 91,944,421
Output size: 15.03 GB


,started_at,ended_at,start_station_id,start_station_name,start_lat,start_lng,end_station_id,end_station_name,end_lat,end_lng,member_casual,birth_year,gender,trip_duration_s
0,2013-06-01 00:00:01,2013-06-01 00:11:36,444,Broadway & W 24 St,40.742354,-73.989151,434,9 Ave & W 18 St,40.743174,-74.003664,member,1983.0,1,695
1,2013-06-01 00:00:08,2013-06-01 00:11:41,444,Broadway & W 24 St,40.742354,-73.989151,434,9 Ave & W 18 St,40.743174,-74.003664,member,1984.0,1,693
2,2013-06-01 00:00:44,2013-06-01 00:35:03,406,Hicks St & Montague St,40.695128,-73.995951,406,Hicks St & Montague St,40.695128,-73.995951,casual,NaN,0,2059
3,2013-06-01 00:01:04,2013-06-01 00:03:07,475,E 15 St & Irving Pl,40.735243,-73.987586,262,Washington Park,40.691782,-73.973730,member,1960.0,1,123
4,2013-06-01 00:01:22,2013-06-01 00:26:43,2008,Little West St & 1 Pl,40.705693,-74.016777,310,State St & Smith St,40.689269,-73.989129,member,1983.0,1,1521


## 4. Initial data inspection (Preliminary EDA)

## 4.2 Missing values and numeric ranges

Scans the combined file in chunks to report per-column missing values and numeric ranges without loading all trips into memory. This is a preliminary EDA check; data cleaning and aggregation are handled in later sections.

In [4]:
numeric_columns = ["start_lat", "start_lng", "end_lat", "end_lng", "birth_year", "gender", "trip_duration_s"]

if not OUTPUT_PATH.exists():
    raise FileNotFoundError("Run Section 3 assembly before the preliminary quality scan.")

missing_counts = pd.Series(0, index=UNIFIED_COLUMNS, dtype="int64")
numeric_summary = {
    column: {"valid_count": 0, "min": float("inf"), "max": float("-inf"), "sum": 0.0}
    for column in numeric_columns
}
quality_row_count = 0

for quality_chunk in pd.read_csv(OUTPUT_PATH, chunksize=CHUNK_SIZE, low_memory=False):
    quality_row_count += len(quality_chunk)
    missing_counts = missing_counts.add(
        quality_chunk.reindex(columns=UNIFIED_COLUMNS).isna().sum(), fill_value=0
    ).astype("int64")

    for column in numeric_columns:
        values = pd.to_numeric(quality_chunk[column], errors="coerce").dropna()
        if values.empty:
            continue
        stats = numeric_summary[column]
        stats["valid_count"] += int(values.count())
        stats["min"] = min(stats["min"], float(values.min()))
        stats["max"] = max(stats["max"], float(values.max()))
        stats["sum"] += float(values.sum())

print(f"Rows scanned: {quality_row_count:,}")
print("Missing values by column:")
display(missing_counts.rename("missing_count").to_frame())

numeric_rows = []
for column, stats in numeric_summary.items():
    valid_count = stats["valid_count"]
    numeric_rows.append({
        "column": column,
        "valid_count": valid_count,
        "missing_or_non_numeric": quality_row_count - valid_count,
        "min": stats["min"] if valid_count else pd.NA,
        "mean": stats["sum"] / valid_count if valid_count else pd.NA,
        "max": stats["max"] if valid_count else pd.NA,
    })
print("Numeric column ranges:")
display(pd.DataFrame(numeric_rows).set_index("column"))

Rows scanned: 91,944,421
Missing values by column:


,missing_count
started_at,0
ended_at,0
start_station_id,2677
start_station_name,2677
start_lat,0
start_lng,0
end_station_id,20736
end_station_name,20736
end_lat,18059
end_lng,18059


Numeric column ranges:


,valid_count,missing_or_non_numeric,min,mean,max
column,,,,,
start_lat,91944421,0,0.000000,40.736670,4.550636e+01
start_lng,91944421,0,-74.034234,-73.985345,0.000000e+00
end_lat,91926362,18059,0.000000,40.736312,4.550636e+01
end_lng,91926362,18059,-74.085849,-73.985442,0.000000e+00
birth_year,85714946,6229475,1857.000000,1978.394283,2.003000e+03
gender,91944421,0,0.000000,1.120853,2.000000e+00
trip_duration_s,91944421,0,60.000000,965.523743,1.951005e+07


## 4.3 Data anomalies and cleaning rules (2013–2019)

- **Schema fields not retained:** This period uses legacy schemas without `ride_id` or `rideable_type`; `bike_id` is intentionally omitted as unused. They are not part of the unified output.
- **Other missing values:** Rows missing timestamps, station IDs/names, coordinates, or rider type are removed because those fields are required for station/day analysis.
- **Coordinates:** Require both start and end coordinates to fall within broad NYC bounds: latitude 39–42 and longitude -76 to -72.
- **Birth year:** Rows with a recorded birth year below 1900 are removed as implausible. Missing birth years remain valid.
- **Trip duration:** Remove trips with duration at or below 0 seconds or above 1 week (604,800 seconds).

Section 5 applies these filters to a separate cleaned output. The original combined file is kept unchanged.

## 5. Clean trip-level data

Apply the documented EDA filters chunk by chunk and write a separate plain CSV. Era-specific missing columns remain missing; the raw unified source is not modified.

In [5]:
CLEANED_OUTPUT_PATH = DATA_DIR / "processed" / "citibike_trips_cleaned_2013_2019.csv"
CLEANING_TEMP_PATH = CLEANED_OUTPUT_PATH.with_suffix(CLEANED_OUTPUT_PATH.suffix + ".tmp")
MAX_TRIP_DURATION_SECONDS = 7 * 24 * 60 * 60

cleaning_counts = {
    "input_rows": 0,
    "missing_required_fields": 0,
    "invalid_nyc_coordinates": 0,
    "birth_year_before_1900": 0,
    "invalid_trip_duration": 0,
    "kept_rows": 0,
}
required_fields = [
    "started_at", "ended_at", "start_station_id", "start_station_name",
    "start_lat", "start_lng", "end_lat", "end_lng", "member_casual",
]
CLEANED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
CLEANING_TEMP_PATH.unlink(missing_ok=True)
header_written = False

try:
    with open(CLEANING_TEMP_PATH, "w", encoding="utf-8", newline="") as cleaned_file:
        for clean_chunk in pd.read_csv(OUTPUT_PATH, chunksize=CHUNK_SIZE, low_memory=False):
            cleaning_counts["input_rows"] += len(clean_chunk)
            for column in ("start_lat", "start_lng", "end_lat", "end_lng", "birth_year", "trip_duration_s"):
                clean_chunk[column] = pd.to_numeric(clean_chunk[column], errors="coerce")
            clean_chunk["started_at"] = pd.to_datetime(clean_chunk["started_at"], format="mixed", errors="coerce")
            clean_chunk["ended_at"] = pd.to_datetime(clean_chunk["ended_at"], format="mixed", errors="coerce")

            missing_mask = clean_chunk[required_fields].isna().any(axis=1)
            cleaning_counts["missing_required_fields"] += int(missing_mask.sum())
            clean_chunk = clean_chunk.loc[~missing_mask].copy()

            coordinate_mask = (
                clean_chunk["start_lat"].between(39, 42)
                & clean_chunk["end_lat"].between(39, 42)
                & clean_chunk["start_lng"].between(-76, -72)
                & clean_chunk["end_lng"].between(-76, -72)
            )
            cleaning_counts["invalid_nyc_coordinates"] += int((~coordinate_mask).sum())
            clean_chunk = clean_chunk.loc[coordinate_mask].copy()

            invalid_birth_year = clean_chunk["birth_year"].notna() & clean_chunk["birth_year"].lt(1900)
            cleaning_counts["birth_year_before_1900"] += int(invalid_birth_year.sum())
            clean_chunk = clean_chunk.loc[~invalid_birth_year].copy()

            invalid_duration = (
                clean_chunk["trip_duration_s"].isna()
                | clean_chunk["trip_duration_s"].le(0)
                | clean_chunk["trip_duration_s"].gt(MAX_TRIP_DURATION_SECONDS)
            )
            cleaning_counts["invalid_trip_duration"] += int(invalid_duration.sum())
            clean_chunk = clean_chunk.loc[~invalid_duration]

            cleaning_counts["kept_rows"] += len(clean_chunk)
            clean_chunk.to_csv(cleaned_file, index=False, header=not header_written)
            header_written = True

    if not header_written:
        raise ValueError("Cleaning produced no rows; the cleaned output was not written.")
    CLEANING_TEMP_PATH.replace(CLEANED_OUTPUT_PATH)
except Exception:
    CLEANING_TEMP_PATH.unlink(missing_ok=True)
    raise

removed_rows = cleaning_counts["input_rows"] - cleaning_counts["kept_rows"]
print(f"Cleaned output: {CLEANED_OUTPUT_PATH}")
print(f"File size: {CLEANED_OUTPUT_PATH.stat().st_size / 1_000_000_000:.2f} GB")
print(f"Rows kept: {cleaning_counts['kept_rows']:,} / {cleaning_counts['input_rows']:,}")
print(f"Rows removed: {removed_rows:,}")
print("Sequential removal counts:")
for rule, count in cleaning_counts.items():
    if rule not in {"input_rows", "kept_rows"}:
        print(f"  {rule.replace('_', ' '):30} {count:>12,}")

Cleaned output: c:\Users\lemme\OneDrive\Desktop\2025_School\deep learning\Ctrl-Alt-Defeat\NYC Citi Bike System Data\data\processed\citibike_trips_cleaned_2013_2019.csv
File size: 15.06 GB
Rows kept: 91,854,873 / 91,944,421
Rows removed: 89,548
Sequential removal counts:
  missing required fields              72,516
  invalid nyc coordinates                 369
  birth year before 1900               13,065
  invalid trip duration                 3,598
